# Bus Stop Data Cleaning

This notebook downloads bus stop data from the LTA DataMall API and standardizes the output for downstream geospatial joins.

## Cleaning objective
- load the LTA API key from the environment
- download the full bus stop catalogue
- standardize coordinate and description field names
- export the cleaned dataset to the project `data/cleaned` folder


In [4]:
# ============================================================
# 1. Imports and configuration
# ============================================================

import os
from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv

load_dotenv()

OUTPUT_FILE = Path("../../data/cleaned/bus_stops_cleaned.csv")
LTA_KEY = os.getenv("LTA_KEY")


## Download and standardize bus stop records

The helper function below pulls all pages from the API, combines the rows into a single dataframe, and renames key columns to match the rest of the project.


In [5]:
# ============================================================
# 2. Helper function
# ============================================================

def get_bus_stops(account_key: str) -> pd.DataFrame:
    """Fetch the complete bus stop catalogue from LTA DataMall."""
    if not account_key:
        raise ValueError("Missing `LTA_KEY`. Add it to your environment before running this notebook.")

    url = "https://datamall2.mytransport.sg/ltaodataservice/BusStops"
    headers = {"AccountKey": account_key, "accept": "application/json"}
    rows = []
    skip = 0

    while True:
        response = requests.get(
            url,
            headers=headers,
            params={"$skip": skip},
            timeout=30,
        )
        response.raise_for_status()
        chunk = response.json().get("value", [])

        if not chunk:
            break

        rows.extend(chunk)
        skip += 500

    bus_df = pd.DataFrame(rows)
    if bus_df.empty:
        return bus_df

    return bus_df.rename(
        columns={
            "Latitude": "lat",
            "Longitude": "lon",
            "Description": "bus_stop_desc",
        }
    )


## Create cleaned output

This step runs the download, previews the cleaned dataframe, and writes the result to disk.


In [6]:
# ============================================================
# 3. Build cleaned dataset
# ============================================================

bus_df = get_bus_stops(account_key=LTA_KEY)
print("Bus stop records:", len(bus_df))
bus_df.head()


Bus stop records: 5200


,BusStopCode,RoadName,bus_stop_desc,lat,lon
0,01012,Victoria St,Hotel Grand Pacific,1.296848,103.852536
1,01013,Victoria St,St. Joseph's Ch,1.297710,103.853225
2,01019,Victoria St,Bras Basah Cplx,1.296990,103.853022
3,01029,Nth Bridge Rd,Opp Natl Lib,1.296673,103.854414
4,01039,Nth Bridge Rd,Bugis Cube,1.298208,103.855491


In [7]:
# ============================================================
# 4. Save cleaned dataset
# ============================================================

OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
bus_df.to_csv(OUTPUT_FILE, index=False)
print(f"Saved cleaned bus stop data to {OUTPUT_FILE.resolve()}")


Saved cleaned bus stop data to /Users/celine/Documents/GitHub/DSA4264-Public-Policy-and-Society/data/cleaned/bus_stops_cleaned.csv
